# Ders 12: Ölçekte Verimli Eğitim ve Çıkarım

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: Ders 2 (parametre ve FLOP muhasebesi), Ders 3 (KV önbelleği), Ders 5 (niceleme).

Son defter, neyin gerçekten eğitilebilir ve servis edilebilir olduğunu belirleyen kısıt hakkında:
**donanım**. Buradaki tekniklerin neredeyse hepsi aynı olgunun cevabıdır — aritmetik ucuzdur, bellek
hareketi değildir. Sayısal formatları, aktivasyon belleğini, paralelliğin üç eksenini, haberleşme
maliyetini ve boştaki bir hızlandırıcıda bile üretimin neden yavaş olduğunu açıklayan çatı çizgisi
(roofline) analizini adım adım işliyoruz.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. Sayısal Formatlar: Aralık, Kesinliği Yener

Bir kayan nokta formatı bitlerini üs (aralık) ile mantis (kesinlik) arasında paylaştırır. Üç 16-bit
seçenek yalnızca bu paylaşımda farklılaşır:

| Format | Üs | Mantis | Maks | En küçük normal | Ondalık basamak |
|---|---|---|---|---|---|
| fp32 | 8 | 23 | $3.4\times10^{38}$ | $1.2\times10^{-38}$ | ~7 |
| fp16 | 5 | 10 | $6.6\times10^{4}$ | $6.1\times10^{-5}$ | ~3 |
| bf16 | 8 | 7 | $3.4\times10^{38}$ | $1.2\times10^{-38}$ | ~2 |

bf16, fp32'nin üssünü korur ve mantis bitlerinden feragat eder. Derin öğrenme için doğru takas
budur: gradyanlar pek çok kat büyüklüğe yayılır; kesinlik kaybetmek biraz gürültüye mal olur, aralık
kaybetmek ise NaN'a. fp16 eğitimi **kayıp ölçekleme** gerektirir — kaybı büyük bir sabitle çarpın ki
küçük gradyanlar hayatta kalsın, sonra güncellemeden önce geri bölün — bf16 ise genellikle hiçbir şey
gerektirmez.


In [ ]:
def quantize_float(x, exp_bits, man_bits):
    max_exp = 2**(exp_bits-1) - 1
    min_exp = -(2**(exp_bits-1) - 2)
    out = np.zeros_like(x, dtype=float)
    nz = x != 0
    e = np.floor(np.log2(np.abs(x[nz])))
    e = np.clip(e, min_exp, max_exp)
    scale = 2.0**(e - man_bits)
    q = np.round(x[nz]/scale)*scale
    lim = (2 - 2.0**-man_bits)*2.0**max_exp
    out[nz] = np.clip(q, -lim, lim)
    out[nz] = np.where(np.abs(x[nz]) < 2.0**min_exp, 0.0, out[nz])     # alt taşmada sıfırla
    return out

# Gerçekçi bir gradyan dağılımı pek çok kat büyüklüğe yayılır
g = 10.0**np.random.normal(-4.5, 1.6, 200000) * np.random.choice([-1, 1], 200000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(np.log10(np.abs(g)), bins=80, color="steelblue", alpha=0.85)
for v, lab, c in [(np.log10(6.1e-5), "fp16 en küçük normal", "crimson"), (np.log10(1.2e-38), "bf16/fp32 en küçük", "seagreen")]:
    axes[0].axvline(v, ls="--", lw=2, c=c, label=lab)
axes[0].set_xlim(-12, 2); axes[0].set_xlabel("log10 |gradyan|"); axes[0].set_ylabel("sayı")
axes[0].set_title("Gradyanlar fp16'nın alt taşma sınırının yakınında yaşıyor"); axes[0].legend(fontsize=8)

fp16 = quantize_float(g, 5, 10)
bf16 = quantize_float(g, 8, 7)
lost = {"fp16": np.mean(fp16 == 0), "bf16": np.mean(bf16 == 0)}
err  = {"fp16": np.mean(np.abs(fp16-g)[fp16 != 0]/np.abs(g)[fp16 != 0]),
        "bf16": np.mean(np.abs(bf16-g)[bf16 != 0]/np.abs(g)[bf16 != 0])}
axes[1].bar(list(lost), list(lost.values()), color=["#6baed6", "#08519c"])
axes[1].set_ylabel("sıfıra düşürülen gradyan oranı"); axes[1].set_title("Aralık başarısızlığı (alt taşma)")
for i, (k, v) in enumerate(lost.items()):
    axes[1].text(i, v, f"{v:.1%}", ha="center", va="bottom", fontsize=10)

# Kayıp ölçekleme fp16'yı kurtarır
scales = 2.0**np.arange(0, 20, 2)
frac_lost   = [np.mean(quantize_float(g*s, 5, 10) == 0) for s in scales]
frac_overfl = [np.mean(np.abs(quantize_float(g*s, 5, 10)) >= 6.5e4) for s in scales]
axes[2].semilogx(scales, frac_lost, "o-", lw=2, base=2, label="alt taşıp sıfırlandı")
axes[2].semilogx(scales, frac_overfl, "s-", lw=2, base=2, label="üst taşıp sonsuz oldu")
axes[2].set_xlabel("kayıp ölçeği"); axes[2].set_ylabel("gradyan oranı")
axes[2].set_title("Kayıp ölçekleme: alt ve üst taşma arasındaki pencere")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

for k in lost:
    print(f"{k}: gradyanların {lost[k]:6.2%} kadarı alt taşıyor, kalanın ortalama göreli hatası {err[k]:.2e}")
print("\nDinamik kayıp ölçekleme: sonsuz görünene dek ölçeği artır, geri çekil ve o adımı atla.")


## 2. Aktivasyon Belleği ve Gradyan Kontrol Noktaları

Geri yayılım ileri geçişteki aktivasyonlara ihtiyaç duyar. Her biri $M$ bayt saklayan $L$ katman için
bu $O(LM)$ eder — ve transformer'larda aktivasyon belleği genellikle parametre belleğini fazlasıyla
aşar.

**Gradyan kontrol noktası (checkpointing)** aktivasyonların yalnızca bir altkümesini saklar ve geri
geçiş sırasında geri kalanını yeniden hesaplar. Her $k$. katmanı saklamak $O(L/k + k)$ bellek verir;
bu $k=\sqrt{L}$'de minimumdur ve fazladan bir ileri geçiş (~%33 daha çok hesap) karşılığında
$O(\sqrt{L})$ bellek sağlar.


In [ ]:
L = 64
k = np.arange(1, L+1)
mem = L/k + k                                     # kontrol noktaları + yeniden hesaplanan parça
best = int(k[np.argmin(mem)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(k, mem, lw=2)
axes[0].axvline(best, ls="--", c="crimson", label=f"k* = {best} ~ sqrt(L) = {np.sqrt(L):.0f}")
axes[0].set_xlabel("her k katmanda bir kontrol noktası"); axes[0].set_ylabel("tepe aktivasyon belleği (M biriminde)")
axes[0].set_title(f"L = {L} katman"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

Ls = np.arange(8, 200)
axes[1].plot(Ls, Ls, lw=2, label="kontrol noktası yok  O(L)")
axes[1].plot(Ls, 2*np.sqrt(Ls), lw=2, label="karekök kontrol noktası  O(sqrt L)")
axes[1].set_xlabel("katman sayısı L"); axes[1].set_ylabel("aktivasyon belleği")
axes[1].set_title("Bellek - derinlik"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

strategies = ["yok", "her 8. katman", "sqrt(L)", "her katman"]
memory  = [L, L/8 + 8, 2*np.sqrt(L), 2.0]
compute = [1.0, 1.12, 1.33, 2.0]
x = np.arange(4); w = 0.35
axes[2].bar(x-w/2, np.array(memory)/L, w, label="göreli bellek")
axes[2].bar(x+w/2, compute, w, label="göreli hesap")
axes[2].set_xticks(x); axes[2].set_xticklabels(strategies, fontsize=9)
axes[2].set_title("Bellek-hesap takas oranı"); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()

# Somut transformer aktivasyon muhasebesi
def activation_bytes(b, n, d, L, heads, bytes_per=2):
    per_layer = b*n*d*bytes_per*10 + b*heads*n*n*bytes_per     # artık akış terimleri + dikkat matrisi
    return per_layer*L

for b, n in [(8, 2048), (8, 8192), (32, 2048)]:
    tot = activation_bytes(b, n, 4096, 32, 32)
    print(f"yığın {b:3d}, dizi {n:5d}: aktivasyon {tot/1e9:7.1f} GB   "
          f"karekök kontrol noktasıyla {tot/1e9*2/np.sqrt(32):6.1f} GB")


## 3. Paralelliğin Üç Ekseni

| Tür | Bölünen | Adım başına haberleşme | Ana sınır |
|---|---|---|---|
| **Veri** | yığın | tüm gradyanların all-reduce'u | model tek cihaza sığmalı |
| **Tensör** | katman içindeki matrisler | **katman başına iki kez** all-reduce | çok hızlı ara bağlantı gerekir (düğüm içi) |
| **Boru hattı** | katmanlar, cihazlara dağıtılır | yalnızca aşama sınırlarındaki aktivasyonlar | boru hattı kabarcığı |
| **Dizi / bağlam** | dizi boyutu | dikkat, cihazlar arası anahtarlara ihtiyaç duyar | uzun bağlamlı eğitim |

Gerçek sistemler bunları birleştirir — "3B paralellik": ara bağlantının hızlı olduğu düğüm içinde
tensör paralelliği, düğümler arasında boru hattı paralelliği ve bunların üzerinde veri paralelliği.

Boru hattının karakteristik maliyeti **kabarcıktır** (bubble): $P$ aşama ve $m$ mikro-yığınla,
cihazların boş geçirdiği zaman oranı

$$\text{kabarcık} = \frac{P-1}{m+P-1}$$

kadardır. Daha çok mikro-yığın bunu küçültür; bedeli, aynı anda havada olan daha fazla aktivasyon
belleğidir.


In [ ]:
P = np.arange(2, 33)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for m, c in zip([1, 2, 4, 8, 32], ["#deebf7", "#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[0].plot(P, (P-1)/(m+P-1), lw=2, c=c, label=f"{m} mikro-yığın")
axes[0].set_xlabel("boru hattı aşaması P"); axes[0].set_ylabel("boş geçen zaman oranı")
axes[0].set_title("Boru hattı kabarcığı"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Boru hattı çizelgesi (GPipe tarzı: önce tüm ileri, sonra tüm geri geçişler)
Pp, mm = 4, 6
sched = np.full((Pp, 2*(mm+Pp-1)), np.nan)
for s in range(Pp):
    for i in range(mm):
        sched[s, s+i] = 0                                   # ileri
        sched[s, (mm+Pp-1) + (Pp-1-s) + i] = 1              # geri
axes[1].imshow(sched, aspect="auto", cmap="coolwarm", interpolation="nearest")
axes[1].set_xlabel("zaman adımı"); axes[1].set_ylabel("boru hattı aşaması")
axes[1].set_title(f"GPipe çizelgesi: beyaz = boş (%{(P-1)[Pp-2]/(mm+Pp-1)*100:.0f} kabarcık)")

# ZeRO aşamaları: Adam ile 7B model için cihaz başına bellek
N, world = 7e9, 8
comps = {
    "temel (DDP)":     [N*2, N*2, N*12],
    "ZeRO-1 (optimizasyon durumu)": [N*2, N*2, N*12/world],
    "ZeRO-2 (+ gradyanlar)":   [N*2, N*2/world, N*12/world],
    "ZeRO-3 (+ parametreler)":  [N*2/world, N*2/world, N*12/world],
}
labels = list(comps); vals = np.array(list(comps.values()))/1e9
bottom = np.zeros(len(labels))
for i, part in enumerate(["parametreler (bf16)", "gradyanlar (bf16)", "optimizasyon durumu (fp32)"]):
    axes[2].bar(labels, vals[:, i], bottom=bottom, label=part)
    bottom += vals[:, i]
axes[2].axhline(80, ls="--", c="crimson", label="80 GB cihaz")
axes[2].set_ylabel("cihaz başına bellek (GB)"); axes[2].set_title(f"{world} cihaza ZeRO parçalama")
axes[2].legend(fontsize=8); plt.setp(axes[2].get_xticklabels(), rotation=12, fontsize=8)
plt.tight_layout(); plt.show()

for k_, v in comps.items():
    print(f"{k_:26s} {sum(v)/1e9:7.1f} GB / cihaz")


## 4. Haberleşme: Halka All-Reduce Neden Varsayılan?

Naif bir all-reduce, her cihazın tüm gradyanını bir parametre sunucusuna gönderir: tek bir bağlantı
üzerinde $O(N \cdot P)$ trafik. **Halka all-reduce** gradyanı $P$ parçaya böler ve iki aşamada
halkada dolaştırır (reduce-scatter, sonra all-gather). Her cihaz tam olarak

$$2\,\frac{P-1}{P}\,N \ \text{ bayt}$$

gönderir; bu **$P$'den bağımsızdır** — veri paralelliğinin ölçeklenebilmesini sağlayan özellik budur.
Geriye kalan maliyet gecikmedir, $2(P-1)$ sıçrama; çok küçük modellerin kötü paralelleşmesinin nedeni
de budur.

Pratikteki teknik **örtüşmedir** (overlap): son katmanların gradyanları önce hazır olur, dolayısıyla
onların all-reduce'u erken katmanlar hâlâ hesaplanırken başlatılır (bucketing). Adım başına yeterli
hesap varsa haberleşme tamamen hesabın arkasına gizlenir.


In [ ]:
N_bytes = 7e9*2
bw, latency = 200e9, 5e-6          # 200 GB/s ara bağlantı, sıçrama başına 5 us
Ps = np.arange(2, 65)

naive = N_bytes*(Ps-1)/bw
ring  = 2*(Ps-1)/Ps*N_bytes/bw + 2*(Ps-1)*latency

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(Ps, naive, lw=2, label="parametre sunucusu  O(N P)")
axes[0].plot(Ps, ring, lw=2, label="halka all-reduce  O(N)")
axes[0].set_yscale("log"); axes[0].set_xlabel("cihaz sayısı P"); axes[0].set_ylabel("all-reduce başına süre (s, log)")
axes[0].set_title("Haberleşme hacmi"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

# Örtüşmeli ve örtüşmesiz ölçekleme verimliliği
compute_time = np.array([0.02, 0.1, 0.5, 2.0])
for ct, c in zip(compute_time, ["#deebf7", "#c6dbef", "#6baed6", "#08306b"]):
    comm = 2*(Ps-1)/Ps*N_bytes/bw + 2*(Ps-1)*latency
    eff_seq = ct/(ct + comm)
    axes[1].plot(Ps, eff_seq, lw=2, c=c, label=f"hesap {ct}s/adım")
axes[1].set_xlabel("cihaz sayısı P"); axes[1].set_ylabel("ölçekleme verimliliği")
axes[1].set_title("Örtüşme olmadan"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

for ct, c in zip(compute_time, ["#deebf7", "#c6dbef", "#6baed6", "#08306b"]):
    comm = 2*(Ps-1)/Ps*N_bytes/bw + 2*(Ps-1)*latency
    exposed = np.maximum(comm - 0.9*ct, 0)                    # haberleşmenin %90'ı hesabın arkasında gizli
    axes[2].plot(Ps, ct/(ct + exposed), lw=2, c=c, label=f"hesap {ct}s/adım")
axes[2].set_xlabel("cihaz sayısı P"); axes[2].set_ylabel("ölçekleme verimliliği")
axes[2].set_title("Gradyan paketleme / örtüşme ile"); axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print(f"200 GB/s bağlantı üzerinde 7B modelin halka all-reduce'u: {2*63/64*N_bytes/bw*1000:.0f} ms, "
      f"cihaz sayısından bağımsız.")


## 5. Çıkarım: Ön Dolum Hesap Sınırlı, Kod Çözme Bellek Sınırlıdır

Üretimin donanım davranışı tamamen farklı iki aşaması vardır.

**Ön dolum (prefill)** istemin tamamını tek seferde işler: büyük matris–matris çarpımları, yüksek
**aritmetik yoğunluk** (taşınan bayt başına FLOP), hesap sınırlı ve verimli.

**Kod çözme (decode)** her seferinde tek bir token üretir: matris–**vektör** çarpımları; yani tek bir
çarp-topla işlemi için her ağırlık bellekten okunur. Aritmetik yoğunluk $\approx 1$'dir ve hızlandırıcı
tepe kapasitesinin çok küçük bir kısmında çalışır. Kod çözme hızı esasen şudur:

$$\text{token/s} \approx \frac{\text{bellek bant genişliği}}{\text{token başına okunan ağırlık baytı}}.$$

Çözüm yığınlamadır — $B$ dizi aynı ağırlık okumasını paylaşır ve yoğunluğu $B$ katına çıkarır — ta ki
KV önbelleği belleği tüketene kadar. Bu, çatı çizgisi modelinin en sonuç doğurucu hâlidir.


In [ ]:
peak_flops, bw = 1000e12, 3.35e12         # 1 PFLOP/s, 3.35 TB/s
ridge = peak_flops/bw

intensity = np.logspace(-1, 4, 400)
attainable = np.minimum(peak_flops, bw*intensity)

N_params, bytes_w = 7e9, 2
batches = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].loglog(intensity, attainable/1e12, lw=2, c="k")
axes[0].axvline(ridge, ls="--", c="gray")
axes[0].text(ridge*1.15, 20, f"kırılma noktası\n{ridge:.0f} FLOP/bayt", fontsize=8)
axes[0].scatter(batches, np.minimum(peak_flops, bw*batches)/1e12, c="crimson", s=40, zorder=4,
                label="kod çözme, yığın = 1..256")
axes[0].scatter([2048], [np.minimum(peak_flops, bw*2048)/1e12], c="seagreen", s=80, marker="s",
                zorder=4, label="ön dolum (2048 token'lık istem)")
axes[0].set_xlabel("aritmetik yoğunluk (FLOP/bayt)"); axes[0].set_ylabel("ulaşılabilir TFLOP/s")
axes[0].set_title("Çatı çizgisi"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which="both")

tok_s = bw/(N_params*bytes_w)*batches
axes[1].loglog(batches, tok_s, "o-", lw=2, base=2, label="toplam token/s")
axes[1].loglog(batches, tok_s/batches, "s-", lw=2, base=2, label="istek başına token/s")
axes[1].set_xlabel("yığın boyutu"); axes[1].set_ylabel("verim")
axes[1].set_title("Yığınlama gecikmeyi verimle takas eder")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")

# Yığın boyutunu sınırlayan şey KV önbelleğidir
ctx = np.array([1024, 4096, 16384, 65536])
kv_per_seq = 2*ctx*32*32*128*2/1e9         # 2 * n * L * H * d_head * bayt
free_gb = 80 - 14
for c_, col in zip(ctx, ["#deebf7", "#c6dbef", "#6baed6", "#08306b"]):
    kv = 2*c_*32*32*128*2/1e9
    axes[2].bar(str(c_), free_gb/kv, color=col)
axes[2].set_xlabel("bağlam uzunluğu"); axes[2].set_ylabel("maks eşzamanlı dizi")
axes[2].set_yscale("log"); axes[2].set_title("KV önbelleği yığın boyutunu sınırlar")
plt.tight_layout(); plt.show()

print(f"yığın 1 ile kod çözme : {bw/(N_params*bytes_w):7.1f} token/s, "
      f"tepe kapasitenin {bw*1/peak_flops:.3%} kadarı")
print(f"yığın 64 ile kod çözme: {bw/(N_params*bytes_w)*64:7.1f} token/s, "
      f"tepe kapasitenin {min(bw*64/peak_flops,1):.1%} kadarı")
for c_, k_ in zip(ctx, kv_per_seq):
    print(f"bağlam {c_:6d}: KV önbelleği {k_:6.2f} GB/dizi -> en fazla {free_gb/k_:6.1f} eşzamanlı dizi")


## 6. Spekülatif Kod Çözme

Kod çözme bant genişliği sınırlı olduğundan, birkaç token'ı doğrulamak neredeyse tek token üretmekle
aynı maliyettedir. Spekülatif kod çözme bunu kullanır: ucuz bir **taslak** model $k$ token önerir,
hedef model bunların hepsini tek bir ileri geçişte puanlar ve bir reddetme-örneklemesi kuralı,
hedefin dağılımıyla tutarlı olan en uzun öneki kabul eder.

Kabul kuralı, çıktı dağılımının **tam olarak** hedef modelinki olmasını garanti eder — bu bir
yaklaşıklama değil, saf bir gecikme optimizasyonudur. Token başına kabul olasılığı $\alpha$ ise hedef
çağrısı başına beklenen token sayısı

$$\mathbb{E}[\text{kabul}] = \frac{1-\alpha^{k+1}}{1-\alpha}$$

olur; hızlanma da bunun $1 + k c$'ye bölümüdür ($c$: taslağın göreli maliyeti).


In [ ]:
def speedup(alpha, k, c):
    expected = (1 - alpha**(k+1))/(1 - alpha) if alpha < 1 else k+1
    return expected/(1 + k*c)

ks = np.arange(1, 13)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for a, col in zip([0.5, 0.7, 0.8, 0.9], ["#deebf7", "#c6dbef", "#6baed6", "#08306b"]):
    axes[0].plot(ks, [speedup(a, k, 0.1) for k in ks], "o-", lw=2, c=col, label=f"kabul oranı {a}")
axes[0].set_xlabel("taslak uzunluğu k"); axes[0].set_ylabel("düz kod çözmeye göre hızlanma")
axes[0].set_title("Taslak maliyeti = hedefin %10'u"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

alphas = np.linspace(0.3, 0.98, 100)
for c_, col in zip([0.02, 0.05, 0.1, 0.25], ["#deebf7", "#c6dbef", "#6baed6", "#08306b"]):
    axes[1].plot(alphas, [max(speedup(a, k, c_) for k in ks) for a in alphas], lw=2, c=col,
                 label=f"taslak maliyeti {c_}")
axes[1].axhline(1, ls="--", c="crimson")
axes[1].set_xlabel("token başına kabul oranı"); axes[1].set_ylabel("ulaşılabilir en iyi hızlanma")
axes[1].set_title("Yavaş bir taslak model işleri kötüleştirebilir"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Kabul/ret sürecinin benzetimi
rng = np.random.default_rng(0)
def simulate(alpha, k, n=20000):
    acc = (rng.random((n, k)) < alpha)
    first_reject = np.where(acc.all(1), k, acc.argmin(1))
    return first_reject + 1                                # hedefin kendi ürettiği token için +1
for a, col in zip([0.6, 0.8, 0.9], ["#c6dbef", "#6baed6", "#08306b"]):
    v = simulate(a, 6)
    axes[2].hist(v, bins=np.arange(1, 9)-0.5, alpha=0.6, color=col, label=f"alpha={a}, ortalama {v.mean():.2f}")
axes[2].set_xlabel("hedef ileri geçişi başına kabul edilen token"); axes[2].set_ylabel("sayı")
axes[2].set_title("k = 6 draft tokens"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'alpha':>6} {'en iyi k':>9} {'hızlanma':>10}")
for a in [0.5, 0.7, 0.8, 0.9, 0.95]:
    best_k = int(ks[np.argmax([speedup(a, k, 0.1) for k in ks])])
    print(f"{a:6.2f} {best_k:7d} {speedup(a, best_k, 0.1):9.2f}x")


## 7. Budama ve Yapılandırılmış Seyreklik

Son kaldıraç ağırlıkları atmaktır. Önemli olan ayrım kaç tanesinin atıldığı değil, **hangi desenle**
atıldığıdır:

- **Yapılandırılmamış** budama (en büyük mutlak değerleri tut) yüksek seyrekliğe az kalite kaybıyla
  ulaşır ama rastgele bir seyreklik deseni, yoğun-matris donanımında hiçbir hızlanma sağlamaz.
  Yalnızca depolama kazandırır.
- **Yapılandırılmış** budama bütün başlıkları, kanalları veya katmanları atar. Kalite açısından çok
  daha zordur ama gerçek ve anında bir hızlanma verir.
- **Yarı-yapılandırılmış** (2:4 — her dört ağırlıktan ikisi sıfır) güncel tensör çekirdeklerinin
  doğrudan desteklediği orta yoldur ve %50 seyreklikte kabaca $2\times$ kazandırır.


In [ ]:
rng = np.random.default_rng(0)
Wm = rng.normal(size=(512, 512))/np.sqrt(512)
x = rng.normal(size=(64, 512))
ref = x @ Wm

def prune_unstructured(W, s):
    t = np.quantile(np.abs(W), s)
    return W*(np.abs(W) >= t)

def prune_structured(W, s):                      # L2 normuna göre tüm çıkış kanallarını at
    keep = int(round((1-s)*W.shape[1]))
    idx = np.argsort(-np.linalg.norm(W, axis=0))[:keep]
    out = np.zeros_like(W); out[:, idx] = W[:, idx]
    return out

def prune_2of4(W):
    Wr = W.reshape(-1, 4).copy()
    order = np.argsort(-np.abs(Wr), axis=1)
    mask = np.zeros_like(Wr, dtype=bool)
    np.put_along_axis(mask, order[:, :2], True, axis=1)
    return (Wr*mask).reshape(W.shape)

sparsities = np.linspace(0.1, 0.95, 18)
err_u = [np.linalg.norm(x@prune_unstructured(Wm, s) - ref)/np.linalg.norm(ref) for s in sparsities]
err_s = [np.linalg.norm(x@prune_structured(Wm, s) - ref)/np.linalg.norm(ref) for s in sparsities]
err_24 = np.linalg.norm(x@prune_2of4(Wm) - ref)/np.linalg.norm(ref)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(sparsities, err_u, "o-", lw=2, label="yapılandırılmamış (mutlak değer)")
axes[0].plot(sparsities, err_s, "s-", lw=2, label="yapılandırılmış (tüm kanallar)")
axes[0].scatter([0.5], [err_24], s=90, c="crimson", zorder=4, label="2:4 yarı-yapılandırılmış")
axes[0].set_xlabel("seyreklik"); axes[0].set_ylabel("göreli çıktı hatası")
axes[0].set_title("Her desenin kalite maliyeti"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].imshow((prune_unstructured(Wm, 0.9)[:64, :64] != 0), cmap="Greys")
axes[1].set_title("Yapılandırılmamış %90: donanım kazancı yok"); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].imshow((prune_2of4(Wm)[:64, :64] != 0), cmap="Greys")
axes[2].set_title("2:4 deseni: tensör çekirdeklerinde 2 kat"); axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout(); plt.show()

i90 = int(np.argmin(np.abs(sparsities-0.9)))
print(f"yapılandırılmamış %90 seyreklik: göreli hata {err_u[i90]:.3f}")
print(f"yapılandırılmış  %90 seyreklik: göreli hata {err_s[i90]:.3f}")
print(f"2:4 (%50 seyreklik)      : göreli hata {err_24:.3f}, ve gerçekten 2 kat hızlı çalışır")
print("\nBu mutlak hatalar kötümserdir: bu, hiç artıklığı olmayan bağımsız rastgele bir matristir;")
print("oysa eğitilmiş ağırlık matrisleri fazlasıyla sıkıştırılabilirdir. Asıl mesele SIRALAMADIR.")


## 8. Hepsini Birleştirmek

Sığmayan bir eğitim koşusu için kontrol listesi:

1. Her yerde **bf16** (kayıp ölçekleme gerekmez); optimizasyonda fp32 ana ağırlıklar.
2. **Gradyan kontrol noktaları** — genellikle aktivasyon belleğinde en büyük tek kazanç.
3. Gerektikçe **ZeRO** ile optimizasyon durumunu, sonra gradyanları, sonra parametreleri parçala.
4. **Düğüm içinde tensör paralelliği**, düğümler arasında boru hattı, üstünde veri paralelliği.
5. Boru hattı kabarcığı küçülene dek mikro-yığın sayısını artır; haberleşmeyi hesapla örtüştür.
6. Ancak bundan sonra daha küçük model ya da daha az token'a yönel.

Servis için:

1. Ağırlıkları nicele (4–8 bit) — kod çözme bant genişliği sınırlı olduğundan bu neredeyse doğrusal
   bir hızlanmadır.
2. Sürekli yığınlama yap; parçalanmayı önlemek için sayfalı (paged) KV önbelleği kullan.
3. Önbelleğin kendisini GQA/MQA ile küçült (Ders 3).
4. Hedef verim değil gecikme ise spekülatif kod çözme uygula.
5. Kalite bütçesi izin veriyorsa damıt ya da buda (2:4).

## 9. Özet

| Kavram | Açıklama |
|---|---|
| **bf16 - fp16** | fp32 ile aynı aralık, karşısında daha çok mantis; gradyanlar için aralık daha önemli |
| **Kayıp ölçekleme** | fp16 gradyanlarını temsil edilebilir pencereye kaydırır |
| **Gradyan kontrol noktası** | ~%33 fazla hesap karşılığında $O(\sqrt L)$ aktivasyon belleği |
| **Veri / tensör / boru hattı** | Yığını / matrisleri / katmanları böler; farklı haberleşme profilleri |
| **Boru hattı kabarcığı** | $(P-1)/(m+P-1)$; daha çok mikro-yığın, daha çok aktivasyon belleği |
| **ZeRO** | Optimizasyon durumunu, gradyanları ve parametreleri veri-paralel süreçlere parçalar |
| **Halka all-reduce** | Cihaz başına $2\frac{P-1}{P}N$ bayt — $P$'den bağımsız |
| **Örtüşme** | Gradyanları paketleyerek haberleşmeyi geri hesabın arkasına gizler |
| **Çatı çizgisi** | Ön dolum hesap sınırlı; kod çözmenin yoğunluğu ≈ 1 ve bant genişliği sınırlı |
| **Yığınlama** | Kod çözme yoğunluğunu doğrusal artırır, KV önbelleği bitene kadar |
| **Spekülatif kod çözme** | Çıktı dağılımı tam olarak aynı; saf bir gecikme optimizasyonu |
| **Seyreklik deseni** | Yapılandırılmamış depolama kazandırır; 2:4 ve yapılandırılmış zaman kazandırır |

---

### Dersin sonu

On iki defter tek bir yayı izliyor: bir model nasıl optimize edilir (1), ne kadar büyük olmalıdır (2),
dikkat nasıl karşılanabilir hâle getirilir (3), temsiller etiketsiz nasıl öğrenilir (4), ucuza nasıl
uyarlanır (5), güvenleri nasıl anlamlı kılınır (6), nasıl üretirler (7), nasıl hizalanırlar (8),
simetri onları nasıl kısıtlar (9), içleri nasıl okunur (10), nerede başarısız olurlar (11) ve donanım
gerçekte neye izin verir (12).
